In [4]:
import pandas as pd
from tqdm import tqdm

import csv
import re
from pathlib import Path

In [ ]:
RAW_ROOT = Path("/home/aniket/Programming/renters/data/cmhc-rental-raw")
CLEAN_ROOT = Path("/home/aniket/Programming/renters/data/cmhc-rental-clean")

SUBFOLDERS = ["avg-rent", "pct-change-rent"]
COLUMNS = ["Year", "Studio", "1 Bedroom", "2 Bedroom", "3 Bedroom+", "Total"]

In [3]:
year_re = re.compile(r"^(\d{4})\s+October\b", re.IGNORECASE)

def _parse_value(raw: str, is_pct: bool):
    if raw is None:
        return None
    value = raw.strip()
    if not value or value == "**":
        return None
    if value == "++":
        return 0.0
    value = value.replace(",", "")
    try:
        num = float(value)
    except ValueError:
        return None
    if is_pct:
        return float(num)
    if num.is_integer():
        return int(num)
    return int(round(num))

def _row_value(row, index):
    if index < len(row):
        return row[index]
    return ""

def parse_cmhc_csv(file_path: Path, is_pct: bool) -> pd.DataFrame:
    with file_path.open("r", encoding="utf-8", errors="replace", newline="") as handle:
        rows = list(csv.reader(handle))

    year_map = {}
    max_year = None
    for row in rows:
        if not row:
            continue
        first_cell = row[0].strip() if len(row) > 0 else ""
        match = year_re.match(first_cell)
        if not match:
            continue
        year = int(match.group(1))
        studio = _parse_value(_row_value(row, 1), is_pct)
        one_bed = _parse_value(_row_value(row, 3), is_pct)
        two_bed = _parse_value(_row_value(row, 5), is_pct)
        three_bed = _parse_value(_row_value(row, 7), is_pct)
        total = _parse_value(_row_value(row, 9), is_pct)
        year_map[year] = [year, studio, one_bed, two_bed, three_bed, total]
        max_year = year if max_year is None else max(max_year, year)

    if max_year is None:
        return pd.DataFrame(columns=COLUMNS)

    data_rows = []
    for year in range(1990, max_year + 1):
        data_rows.append(year_map.get(year, [year, None, None, None, None, None]))

    return pd.DataFrame(data_rows, columns=COLUMNS)

def clean_all_files():
    for subfolder in SUBFOLDERS:
        source_dir = RAW_ROOT / subfolder
        target_dir = CLEAN_ROOT / subfolder
        target_dir.mkdir(parents=True, exist_ok=True)

        files = sorted(source_dir.glob("*.csv"))
        is_pct = subfolder == "pct-change-rent"
        for file_path in tqdm(files, desc=f"Cleaning {subfolder}"):
            df = parse_cmhc_csv(file_path, is_pct)
            out_path = target_dir / file_path.name
            if is_pct:
                for col in COLUMNS[1:]:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                df.to_csv(out_path, index=False, na_rep="", float_format="%.1f")
            else:
                for col in COLUMNS[1:]:
                    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
                df.to_csv(out_path, index=False, na_rep="")

clean_all_files()

Cleaning pct-change-rent: 100%|██████████| 34/34 [00:00<00:00, 482.40it/s]
